<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/Data_transcript_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data transcript analysis

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/Data_transcript_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Google API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Load Git Repository {"form-width":"20%"}
clone_git = True # @param {"type":"boolean"}
if clone_git:
    !git clone https://github.com/PeaceAndLongLife/Analysis-Colab.git

import sys
from pathlib import Path
if '/content/Analysis-Colab/src' not in sys.path:
    sys.path.append('/content/Analysis-Colab/src')

pwd = !pwd
if pwd[0] != '/content/Analysis-Colab/notebooks':
  %cd /content/Analysis-Colab/notebooks

In [ ]:
# @title Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FILE
from google.colab import userdata

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
sys.path.append('src')

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')
# SERVICE_ACCOUNT_FILE = '/content/drive/Shareddrives/Travis Research Drive/Interface 3.0/Admin files/service_account.json'

# 3. Import your file!
from GoogleFunctions import GoogleDocumentManager, extract_file_id
from local_io import read_csv_from_id


## Read in data

data read in will be merged using the `user` column

- consent file
- transcript data

In [ ]:
# @title {"form-width":"20%"}

# @ title ## Input Data Links  {"form-width":"20%"}
# @markdown ---
# Consent data info
# @markdown Read in consent file {"form-width":"20%"}
consent_file_link = "https://drive.google.com/file/d/1wjpPGcbnFkO0tAOvwW5z-Isac4DVzPcv/view?usp=drive_link" # @param {"type":"string"}
consent_file_link_id = extract_file_id(consent_file_link)
show_consent = True # @param {"type":"boolean"}

# Transcript Data info
# @markdown Read in transcript file {"form-width":"20%"}
trans_file_link = "https://drive.google.com/file/d/1Q4G-RJAXd0GgkCyJpA8ajTXTk0hOyQhe/view?usp=drive_link" # @param {"type":"string"}
trans_file_link_id = extract_file_id(trans_file_link)
show_trans = True # @param {"type":"boolean"}
# @markdown ---
# @markdown Combine with 2nd transcript file {"form-width":"20%"}
combine_trans2 = True # @param {"type":"boolean"}
trans2_file_link = "https://drive.google.com/file/d/1rrv-K6od2GkgIw6GXeL-r0RvCaHMkZ9-/view?usp=drive_link" # @param {"type":"string"}
trans2_file_link_id = extract_file_id(trans2_file_link)
show_trans2 = True # @param {"type":"boolean"}
# @markdown ---
# @markdown Combine with 3rd transcript file {"form-width":"20%"}
combine_trans3 = False # @param {"type":"boolean"}
trans3_file_link = "" # @param {"type":"string"}
trans3_file_link_id = extract_file_id(trans3_file_link)
show_trans3 = False # @param {"type":"boolean"}

consent_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, consent_file_link_id, show_consent)
trans_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans_file_link_id, show_trans)
if combine_trans2:
  trans2_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans2_file_link_id, show_trans2)
if combine_trans3:
  trans3_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans3_file_link_id, show_trans3)
# @markdown ---
# display(trans_df)

In [ ]:
# @title Process messages in transcript based on label pattern.
from data_scrub import process_messages, parse_messages, explode_json_messages
import pandas as pd
import re

###
# Merge consent into transcript
combined_df = process_messages(
    consent_df,
    trans_df,
    pattern = r"(.+) - Assignment (\d+), Question: (\d+) - (.+)",
)
if combine_trans2:
  combined2_df = process_messages(
      consent_df,
      trans2_df,
      pattern = r"(.+) - Assignment (\d+) Question (\d+) - (.+)",
  )
if combine_trans3:
  combined3_df = process_messages(
      consent_df,
      trans3_df,
      pattern = r"(.+) - Assignment (\d+) Question (\d+) - (.+)",
  )


In [ ]:
# @title Combine transcripts
import pandas as pd

combined_df = pd.concat([combined_df, combined2_df], ignore_index=True)
display(combined_df.head())

In [ ]:
# @title parse and explode trans
from data_scrub import process_messages, parse_messages, explode_json_messages
import pandas as pd
import re
# Apply the parsing function to the 'all_messages' column
combined_df['all_messages'] = combined_df['all_messages'].apply(parse_messages)

# Count message objects in 'all_messages' and add to 'interactions' column
combined_df['interactions'] = combined_df['all_messages'].apply(lambda x: len(x) / 2)

combined_df = combined_df[combined_df['all_messages'].apply(lambda x: len(x) > 0)]

print("Combined DataFrame with split question columns and interactions:")
# display(combined_df.head())
unexploded_df = combined_df.copy()
combined_df = explode_json_messages(combined_df)
display(unexploded_df.head())

In [ ]:
num_duplicates = combined_df.duplicated().sum()

print(f"Number of duplicate rows in combined_df: {num_duplicates}")

if num_duplicates > 0:
    print("Displaying duplicate rows:")
    display(combined_df[combined_df.duplicated(keep=False)].sort_values(by=list(combined_df.columns)))
else:
    print("No duplicate rows found.")

## Analysis

In [ ]:
# @title Filter by Consent? {"form-width":"20%"}
FilterConsented = True # @param {"type":"boolean"}

if FilterConsented:
  unexploded_df = unexploded_df[unexploded_df['consent'] == True]
  combined_df = combined_df[combined_df['consent'] == True]


In [ ]:
# @title Remove 216
remove_216 = True # @param {"type":"boolean"}
if remove_216:
  unexploded_df = unexploded_df[unexploded_df['Course'] != 'PH216']
  combined_df = combined_df[combined_df['Course'] != 'PH216']

### Analysis Functions

In [ ]:
# @title analyze_stats function
def analyze_hist(df, drop_counts_under_param,y,x):
  # Exclude users with less than drop_counts_under_param unique threads
  df_filtered = df[df[y] >= drop_counts_under_param]


  ### Plot distribution
  import matplotlib.pyplot as plt
  import seaborn as sns

  # Set the style for the plot
  sns.set_style("whitegrid")

  plt.figure(figsize=(10, 6))
  sns.histplot(df_filtered[y], bins=50, kde=True)
  plt.title(f"Distribution of {y} per {x}")
  plt.xlabel(f"{x}")
  # plt.ylabel(f"Number of {y}")
  plt.tight_layout()
  plt.show()
  display(df_filtered[y].describe())


In [ ]:
# @title Bar graph function
def analyze_bar_graph(df, x_col, y_col, title="Bar Graph Distribution", x_label=None, y_label=None, tick_label_size=10):
  import matplotlib.pyplot as plt
  import seaborn as sns

  # Set the style for the plot
  sns.set_style("whitegrid")

  plt.figure(figsize=(12, 6))
  sns.barplot(x=x_col, y=y_col, data=df.sort_values(by=y_col, ascending=False))

  # Add labels and title
  if x_label is None:
    x_label = x_col.replace('_', ' ').title()
  if y_label is None:
    y_label = y_col.replace('_', ' ').title()

  plt.xlabel(x_label)
  plt.ylabel(y_label)
  plt.title(title)

  # Ensure x-axis labels are always horizontal
  plt.xticks(rotation=90)

  # Adjust tick label size
  plt.tick_params(axis='x', labelsize=tick_label_size)
  plt.tick_params(axis='y', labelsize=tick_label_size)

  plt.tight_layout()
  plt.show()

In [ ]:
# @title export_data_csv
def export_data_csv(df, output_df_name,folder_link):
  folder_id = extract_file_id(folder_link)
  gdm = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)
  output_filename = f"{output_df_name}.csv"

  if not folder_id:
    print(f"Error: No target folder ID provided for '{output_filename}'. Please enter a valid Google Drive Folder ID.")
    return
  try:
      uploaded_file_id = gdm.upload_dataframe_to_drive(df, output_filename, folder_id, include_index=False)

      if uploaded_file_id:
          print(f"Successfully saved '{output_df_name}' DataFrame to '{output_filename}'. File ID: {uploaded_file_id}")

      else:
          print(f"Failed to upload '{output_df_name}' DataFrame to '{output_filename}'. See previous error messages.")
  except Exception as e:
      print(f"Error saving '{output_df_name}' DataFrame to '{output_filename}': {e}")
  return


  ###
  # Example implementation

  ## @markdown ---
# export_table = False # @param {"type":"boolean"}
# output_name = "conversations_by_question" # @param {"type":"string","placeholder":"Enter the name of the csv file"}
# target_folder_link = "https://drive.google.com/drive/folders/1Jz9QrU0ZmhCQn4GWh_wa9WtUPzcW2s-s?usp=drive_link" # @param {"type":"string","placeholder":"Enter YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE"}

# if export_table:
#   export_data_csv(df,output_name, target_folder_link)

### Analysis Criteria

In [ ]:
# @title Histogram of all interactions {'run': 'auto', 'form-width': '20%'}

# By default, not dropping any counts under a certain threshold
drop_counts_under = 4 # @param {"type":"slider","min":0,"max":100,"step":1}

# Get unique courses from the unexploded_df
unique_courses = unexploded_df['Course'].unique()

for course in unique_courses:
  print(f"\n--- Histogram for Course: {course} ---")
  # Filter the DataFrame for the current course
  df_course = unexploded_df[unexploded_df['Course'] == course]
  # Call the analyze_hist function to plot the distribution of interactions for the current course
  analyze_hist(df_course, drop_counts_under, 'interactions', f'Interaction Count for {course}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plot
sns.set_style("whitegrid")

plt.figure(figsize=(12, 7))
sns.histplot(data=unexploded_df, x='interactions', hue='Course', bins=50, kde=True)
plt.title('Distribution of Interactions by Course')
plt.xlabel('Number of Interactions')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# @title Make Groups
Course_name = "PH215 (online)" # @param ["PH214 (online)","PH215 (online)"] {"allow-input":true}

df_course = unexploded_df[unexploded_df['Course'] == Course_name]

# x>40
group4 = df_course[(df_course['interactions']>20)][['user', 'Course', 'lab_number', 'question_number', 'interactions']].sort_values(by=['interactions'], ascending=False)
print(f"above 20 interactions. count: {len(group4)}")
display(group4)


print("----\n\n")
# 20<=x<40
group3 = df_course[(df_course['interactions']<=20) & (df_course['interactions']>8)][['user', 'Course', 'lab_number', 'question_number', 'interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Between 8 and 20 interactions. count: {len(group3)}")
display(group3)

print("----\n\n")
# 2<x<=19
group2 = df_course[(df_course['interactions']<=8) & (df_course['interactions']>4)][['user', 'Course', 'lab_number', 'question_number', 'interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Between 4 and 7 interactions. count: {len(group2)}")
display(group2)

print("----\n\n")
# 2<x<=6
group5 = df_course[(df_course['interactions']<=4) & (df_course['interactions']>1)][['user', 'Course', 'lab_number', 'question_number', 'interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Between 1 and 4 interactions. count: {len(group5)}")
display(group5)

print("----\n\n")
# 2>=x
group1 = df_course[df_course['interactions']<=1][['user','Course', 'lab_number', 'question_number', 'interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Less than 1 interactions. count: {len(group1)}")
display(group1)

In [ ]:

# @title export md to Drive {"form-width":"20%"}
drive_folder_link = "https://drive.google.com/drive/folders/1cowYPqZV5SFnTbRcF8TmXkSPkO6fjX28?usp=drive_link" # @param {"type":"string"}
drive_folder_id = extract_file_id(drive_folder_link)
export_md_file = True # @param {"type":"boolean"}
user_num = "256" # @param {"type":"string","placeholder":"Enter user number here"}
notes = "" # @param {"type":"string","placeholder":"Enter Observation notes here"}

table_user = df_course[(df_course['user']==int(user_num))][['user', 'Course', 'lab_number', 'question_number', 'interactions']].sort_values(by=['interactions'], ascending=False)
display(table_user)

if export_md_file:
  # Initialize selected_doc_url and selected_sheet_url if they are not defined
  # This handles the case where the cell is run before a specific lab is selected



  header_blocks=[
        f"User: {user_num}",
        f"Course: {Course_name}",
        "\n---\n\n"
        ]

  header_str = "\n".join(header_blocks)
  table_str = table_user.to_markdown(index=False)

  markdown_text = f"# Transcript Report\n\n{header_str}\n\n## User interactions by lab\n\n{table_str}\n\n Notes:\n\n{notes}"
  filename = f"{user_num}-{Course_name}.md"

  # Upload to Google Drive if drive_folder_id is provided

  gdm = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)
  gdm.export_md(markdown_text, filename, drive_folder_id)
else:
  print("no file exported")

In [ ]:
# @title count conversations by question {"form-width":"20%"}
drop_counts_under = 0 # @param {"type":"slider","min":0,"max":100,"step":1}
# @markdown the x axis represents the question sorted by lab, then number.

current_df = combined_df

thread_counts_df = current_df.groupby(['Course', 'lab_number', 'question_number', 'question', ])['thread'].nunique().reset_index().sort_values(by=['lab_number', 'question_number'])
thread_counts_df.rename(columns={'thread': 'unique_thread_count'}, inplace=True)
# thread_counts_df = thread_counts_df.sort_values(by=['question'])
print("Unique user counts per question:")
# analyze_bar_graph(thread_counts_df, 'question', 'unique_thread_count', 'Conversation Count by Question', tick_label_size=7)
display(thread_counts_df.sort_values(by=['unique_thread_count'], ascending=False))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plot
sns.set_style("whitegrid")

# Preprocess data to count frequencies for barplot
barplot_data = thread_counts_df.groupby(['Course', 'unique_thread_count']).size().reset_index(name='frequency')

plt.figure(figsize=(15, 8))
sns.barplot(data=barplot_data, x='unique_thread_count', y='frequency', hue='Course', dodge=True)
plt.title('Frequency of Unique Thread Counts by Course')
plt.xlabel('Unique Thread Count')
plt.ylabel('Frequency (Number of Questions)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
display(thread_counts_df.sort_values(by=['unique_thread_count'], ascending=False))


In [ ]:
# @title count conversations by lab_number {"form-width":"20%"}
drop_counts_under = 0 # @param {"type":"slider","min":0,"max":100,"step":1}
# @markdown the x axis represents the lab number.

current_df = combined_df

thread_counts_df = current_df.groupby(['Course', 'lab_number'])['thread'].nunique().reset_index().sort_values(by=['lab_number'])
thread_counts_df.rename(columns={'thread': 'unique_thread_count'}, inplace=True)
# thread_counts_df = thread_counts_df.sort_values(by=['question'])
print("Unique user counts per lab:")
sns.barplot(data=thread_counts_df, x='lab_number', y='unique_thread_count', hue='Course', dodge=True)
# analyze_bar_graph(thread_counts_df, 'lab_number', 'unique_thread_count', 'Conversation Count by Question', tick_label_size=7)
display(thread_counts_df.sort_values(by=['unique_thread_count'], ascending=False))


In [ ]:
# @title count users per lab {"form-width":"20%"}

current_df = combined_df

lab_counts_df = current_df.groupby(['Course', 'lab_number'])['user'].nunique().reset_index()
lab_counts_df.rename(columns={'user': 'unique_users'}, inplace=True)
lab_counts_df = lab_counts_df[['Course', 'lab_number','unique_users']].sort_values(by=['unique_users'], ascending=False)
print("Unique thread counts per user and lab_number:")

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.barplot(data=lab_counts_df, x='lab_number', y='unique_users', hue='Course', dodge=True)

# Add numbers on top of the bars
ax = plt.gca()
for container in ax.containers:
    for patch in container.patches:
        height = patch.get_height()
        ax.annotate(f'{int(height)}',
                    xy=(patch.get_x() + patch.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

plt.title('Unique Users by Lab Number')
plt.xlabel('Lab Number')
plt.ylabel('Unique Users')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# analyze_bar_graph(lab_counts_df,'lab_number', 'unique_users', "Unique users by Lab numbers")
display(lab_counts_df)

In [ ]:
# @title count conversations by user {"run":"auto","form-width":"20%"}

drop_counts_under = 0 # @param {"type":"slider","min":0,"max":100,"step":1}
# @markdown "4 groups: x<5, 5<x<13, 13<x<22, 22<x<45, 45<x"
filter_by_lab_number = "False" # @param ["01","02","03","04","05","06","07","08","False"] {"allow-input":true}

# Call the function with the relevant DataFrame and parameter

current_df = combined_df
if filter_by_lab_number != "False":
  current_df = current_df[current_df['lab_number'] < filter_by_lab_number]

lab_counts_df = current_df.groupby(['Course','user'])['thread'].nunique().reset_index()
lab_counts_df.rename(columns={'thread': 'unique_threads'}, inplace=True)

lab_counts_df = lab_counts_df.sort_values(by='unique_threads', ascending=False)

print("Unique thread per user")
import ipywidgets as widgets
from IPython.display import display

# analyze_hist(lab_counts_df, drop_counts_under,'unique_threads','user'
# analyze_hist(lab_counts_df, drop_counts_under,'unique_threads','user')
sns.histplot(data=lab_counts_df, x='unique_threads', hue='Course', bins=50, kde=True)

# analyze_bar_graph(lab_counts_df,'user', 'unique_threads', "Unique threads per user")
display(lab_counts_df[['user','unique_threads']].sort_values(by=['unique_threads'], ascending=False))

print("----\n\n")
# x>=45
group5 = lab_counts_df[lab_counts_df['unique_threads']>=45][['user','unique_threads']].sort_values(by=['unique_threads'], ascending=False)
print(f"greater or equal to 45 conversations. count: {len(group5)}")
display(group5)

print("----\n\n")
# 22<=x<45
group4 = lab_counts_df[(lab_counts_df['unique_threads']<45) & (lab_counts_df['unique_threads']>=22)][['user','unique_threads']].sort_values(by=['unique_threads'], ascending=False)
print(f"Between 22 and 45 conversations. count: {len(group4)}")
display(group4)


print("----\n\n")
# 13<=x<22
group3 = lab_counts_df[(lab_counts_df['unique_threads']<22) & (lab_counts_df['unique_threads']>=13)][['user','unique_threads']].sort_values(by=['unique_threads'], ascending=False)
print(f"Between 13 and 22 conversations. count: {len(group3)}")
display(group3)

print("----\n\n")
# 5<=x<13
group2 = lab_counts_df[(lab_counts_df['unique_threads']<13) & (lab_counts_df['unique_threads']>=5)][['user','unique_threads']].sort_values(by=['unique_threads'], ascending=False)
print(f"Between 5 and 13 conversations. count: {len(group2)}")
display(group2)

print("----\n\n")
# 5>x
group1 = lab_counts_df[lab_counts_df['unique_threads']<5][['user','unique_threads']].sort_values(by=['unique_threads'], ascending=False)
print(f"Less than 5. count: {len(group1)}")
display(group1)


In [ ]:
# @title Total Interactions by Question Plot {"form-width":"20%"}

current_df = combined_df

interactions_by_question = current_df.groupby(['Course', 'lab_number', 'question_number', 'question_text'])['interactions'].sum().reset_index()

# Create a combined question identifier for the x-axis labels if needed, or use existing columns
interactions_by_question['full_question'] = 'Lab' + interactions_by_question['lab_number'] + '-Q' + interactions_by_question['question_number']
analyze_bar_graph(
    df=interactions_by_question,
    x_col='full_question',
    y_col='interactions',
    title='Total Interactions per Question',
    x_label='Question',
    y_label='Total Interactions'
)
interactions_by_question = interactions_by_question.sort_values(by='interactions', ascending=False)
display(interactions_by_question[['full_question','interactions']])


# @markdown ---
export_table = False # @param {"type":"boolean"}
output_name = "conversations_by_question" # @param {"type":"string","placeholder":"Enter the name of the csv file"}
target_folder_link = "https://drive.google.com/drive/folders/1Jz9QrU0ZmhCQn4GWh_wa9WtUPzcW2s-s?usp=drive_link" # @param {"type":"string","placeholder":"Enter YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE"}

if export_table:
  export_data_csv(interactions_by_question[['full_question','interactions']],output_name, target_folder_link)

In [ ]:
# @title Total Interactions by User Plot {"run":"auto","form-width":"20%"}
drop_counts_under = 0 # @param {"type":"slider","min":0,"max":100,"step":1}

current_df = combined_df
current_df=current_df[current_df['Course']=='PH215 (online)']
interactions_by_user = current_df.groupby(['user'])['interactions'].sum().reset_index()

# Create a combined question identifier for the x-axis labels if needed, or use existing columns
# interactions_by_question['full_question'] = 'Lab' + interactions_by_question['lab_number'] + '-Q' + interactions_by_question['question_number']
# analyze_bar_graph(
#     df=interactions_by_user,
#     x_col='user',
#     y_col='interactions',
#     title='Total Interactions per User',
#     x_label='User',
#     y_label='Total Interactions'
# )
analyze_hist(interactions_by_user, drop_counts_under,'interactions','user')
interactions_by_user = interactions_by_user.sort_values(by='interactions', ascending=False)

display(interactions_by_user[['user','interactions']])
#############################
print("----\n\n")
# x>=45
group5 = interactions_by_user[interactions_by_user['interactions']>=229][['user','interactions']].sort_values(by=['interactions'], ascending=False)
print(f"greater or equal to 229 conversations. count: {len(group5)}")
display(group5)

print("----\n\n")
# 22<=x<45
group4 = interactions_by_user[(interactions_by_user['interactions']<229) & (interactions_by_user['interactions']>=104)][['user','interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Between 104 and 229 conversations. count: {len(group4)}")
display(group4)


print("----\n\n")
# 13<=x<22
group3 = interactions_by_user[(interactions_by_user['interactions']<104) & (interactions_by_user['interactions']>=50)][['user','interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Between 50 and 104 conversations. count: {len(group3)}")
display(group3)

print("----\n\n")
# 5<=x<13
group2 = interactions_by_user[(interactions_by_user['interactions']<50) & (interactions_by_user['interactions']>=7)][['user','interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Between 7 and 50 conversations. count: {len(group2)}")
display(group2)

print("----\n\n")
# 5>x
group1 = interactions_by_user[interactions_by_user['interactions']<7][['user','interactions']].sort_values(by=['interactions'], ascending=False)
print(f"Less than 7. count: {len(group1)}")
display(group1)

In [ ]:
# @title export md to Drive {"form-width":"20%"}
drive_folder_link = "https://drive.google.com/drive/folders/1MIIDzYnXAMdBqZGAolzgX_BwSk-zwSrJ?usp=drive_link" # @param {"type":"string"}
drive_folder_id = extract_file_id(drive_folder_link)
export_md_file = True # @param {"type":"boolean"}
interaction_threshold = "661" # @param {"type":"string","placeholder":"Enter threshold here"}
Course_name = "PH215 (online)" # @param {"type":"string","placeholder":"Enter Course here"}
notes = "" # @param {"type":"string","placeholder":"Enter Observation notes here"}

table_interactions = interactions_by_user[interactions_by_user['interactions']>int(interaction_threshold)][['user','interactions']].sort_values(by=['interactions'], ascending=False)
display(table_interactions)

if export_md_file:
  # Initialize selected_doc_url and selected_sheet_url if they are not defined
  # This handles the case where the cell is run before a specific lab is selected


  header_blocks=[
        f"interaction_threshold: {interaction_threshold}",
        "\n---\n\n"
        ]

  header_str = "\n".join(header_blocks)
  table_str = table_interactions.to_markdown(index=False)

  markdown_text = f"# Transcript Report\n\n{header_str}\n\n## User interactions by lab\n\n{table_str}\n\n Notes:\n\n{notes}"
  filename = f"Interactions_above_{interaction_threshold}-{Course_name}.md"

  # Upload to Google Drive if drive_folder_id is provided

  gdm = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)
  gdm.export_md(markdown_text, filename, drive_folder_id)
else:
  print("no file exported")


display(interactions_by_user[interactions_by_user['interactions']>795][['user','interactions']].sort_values(by=['interactions'], ascending=False))

In [ ]:
# @title Total Interactions by Question Plot {"run":"auto","form-width":"20%"}
drop_counts_under = 0 # @param {"type":"slider","min":0,"max":100,"step":1}

current_df = combined_df

interactions_by_lab = current_df.groupby(['Course', 'lab_number'])['interactions'].sum().reset_index()
interactions_by_lab.rename(columns={'interactions': 'interactions_by_lab'}, inplace=True)
# Create a combined question identifier for the x-axis labels if needed, or use existing columns
# interactions_by_question['full_question'] = 'Lab' + interactions_by_question['lab_number'] + '-Q' + interactions_by_question['question_number']
analyze_bar_graph(
    df=interactions_by_lab,
    x_col='lab_number',
    y_col='interactions_by_lab',
    title='Total Interactions per Lab',
    x_label='Lab',
    y_label='Total Interactions'
)
interactions_by_lab = interactions_by_lab.sort_values(by='interactions_by_lab', ascending=False)
display(interactions_by_lab[['lab_number','interactions_by_lab']])

In [ ]:
# @title last lab {"run":"auto","form-width":"20%"}
current_df = combined_df
filter_by_lab_number = "04" # @param ["01","02","03","04","05","06","07","08","False"] {"allow-input":true}


last_lab_per_user = current_df.groupby('user')['lab_number'].max().reset_index()
last_lab_per_user.rename(columns={'lab_number': 'last_lab_number'}, inplace=True)
if filter_by_lab_number != "False":
  last_lab_per_user = last_lab_per_user[last_lab_per_user['last_lab_number'] == filter_by_lab_number]

print("Last (highest) lab number for each user:")
display(last_lab_per_user.sort_values(by='last_lab_number', ascending=False))

In [ ]:
# from google.colab import sheets

# # @title Create Interactive Sheet and optionally move to a specific folder {"form-width":"20%"}
# move_to_target_folder = True # @param {type:"boolean"}
# target_folder_for_sheet_link = "https://drive.google.com/drive/folders/1KeIQKtAkXxhU6mB8uzd8Nw3_Ji7Oroyx?usp=drive_link" # @param {type:"string"}
# sheet_name = "transcript_analysis_Jan14" # @param {"type":"string"}

# # Create the interactive sheet
# sheet_object = sheets.InteractiveSheet(title=sheet_name, include_column_headers=True, df=combined_df)

# # Get the ID of the newly created sheet
# sheet_id = sheet_object.id
# print(f"Interactive Sheet created with ID: {sheet_id}")
# print(f"Interactive Sheet URL: {sheet_object.url}")

# if move_to_target_folder:
#   if 'GDM' not in globals():
#     GDM = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)

#   target_folder_id = extract_file_id(target_folder_for_sheet_link)
#   if target_folder_id:
#     try:
#       # Get current parent(s) to remove from
#       current_parents = GDM.get_file_parent_info(file_id=sheet_id)['parents']
#       current_parent_ids = [p['id'] for p in current_parents]

#       # Move the file by updating its parents
#       GDM.update_file_metadata(
#           file_id=sheet_id,
#           add_parents=target_folder_id,
#           remove_parents=','.join(current_parent_ids)
#       )
#       print(f"Successfully moved Interactive Sheet (ID: {sheet_id}) to folder (ID: {target_folder_id})")
#     except Exception as e:
#       print(f"Error moving sheet to target folder: {e}")
#       print("Please ensure the target folder link is valid and you have appropriate permissions.")
#   else:
#     print(f"Invalid target folder link provided: {target_folder_for_sheet_link}")
